In [2]:
import pandas as pd
import numpy as np

Obsługa **indeksowania hierarchicznego** jest ważnym elementem biblioteki pandas umożliwiającym
przypisanie do jednej osi wielu poziomów indeksowania (przypisanie dwóch lub więcej indeksów).

In [3]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [4]:
data

a  1    0.153052
   2    0.454558
   3    0.514793
b  1    0.204637
   3    0.171363
c  1    0.772480
   2    0.787596
d  2    0.451909
   3    0.382195
dtype: float64

Wyświetloną czytelnie serią, której indeksem jest obiekt MultiIndex.

In [5]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Obiekty o indeksie hierarchicznym obsługują tzw. **indeksowanie częściowe**. Indeksowanie to umożliwia zwięzły wybór podzbioru danych

In [7]:
data["b"]

,0
1,0.204637
3,0.171363


In [8]:
data["b":"c"]

b  1    0.204637
   3    0.171363
c  1    0.772480
   2    0.787596
dtype: float64

In [ ]:
data.loc[["b", "d"]]

## Przykład na danych

In [9]:
# Dane sprzedażowe w dwóch miastach, w dwóch latach
df = pd.DataFrame({
    "miasto":   ["Łódź", "Łódź", "Kraków", "Kraków"],
    "rok":      [2023,   2024,   2023,     2024],
    "sprzedaz": [120,    145,    200,      230]
})
df

,miasto,rok,sprzedaz
0,Łódź,2023,120
1,Łódź,2024,145
2,Kraków,2023,200
3,Kraków,2024,230


In [10]:
df.index

RangeIndex(start=0, stop=4, step=1)

### Zwykły filtrowanie

Za każdym razem: filtr po mieście, filtr po roku, wybór kolumny. Trzy operacje dla jednej wartości.

In [11]:
# Sprzedaż w Łodzi w 2024 - działa, ale rozwlekle:
df[(df.miasto == "Łódź") & (df.rok == 2024)]["sprzedaz"]

,sprzedaz
1,145


In [12]:
# Sprzedaż w Krakowie w 2023 - znowu to samo:
df[(df.miasto == "Kraków") & (df.rok == 2023)]["sprzedaz"]

,sprzedaz
2,200


### MultiIndex - indeks jako "ścieżka"

In [13]:
# Ten sam zbiór, ale z hierarchicznym indeksem:
sprzedaz = pd.Series(
    [120, 145, 200, 230],
    index=[["Łódź", "Łódź", "Kraków", "Kraków"],
           [2023,   2024,   2023,     2024]]
)
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [14]:
# Sam indeks to obiekt MultiIndex:
sprzedaz.index

MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           )

### Indeksowanie - podstawowe wzorce

In [15]:
# Pojedyncza wartość - krotka (miasto, rok):
sprzedaz["Łódź", 2024]

np.int64(145)

In [16]:
# Wszystko dla Łodzi (wybór zewnętrznego poziomu):
sprzedaz["Łódź"]

,0
2023,120
2024,145


In [17]:
# Wszystkie miasta, ale tylko rok 2023 (wewnętrzny poziom):
sprzedaz.loc[:, 2023]

,0
Łódź,120
Kraków,200


In [18]:
# Zakres miast (uwaga: wymaga posortowanego indeksu):
sprzedaz.sort_index().loc["Kraków":"Łódź"]

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64

## Z Series do DataFrame i z powrotem

In [19]:
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [20]:
# unstack - "wypchnięcie" wewnętrznego poziomu do kolumn:
tabela = sprzedaz.unstack()
print(tabela)
print(type(tabela))

        2023  2024
Kraków   200   230
Łódź     120   145
<class 'pandas.core.frame.DataFrame'>


In [21]:
# stack - operacja odwrotna: kolumny wracają do indeksu:
tab_stack = tabela.stack()
print(tab_stack)
print(type(tab_stack))

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64
<class 'pandas.core.series.Series'>


In [22]:
print(df)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


In [23]:
# Konwersja płaskiej ramki na hierarchiczną - set_index:
df_hier = df.set_index(["miasto", "rok"])
print(df_hier)
print(df_hier.index)

             sprzedaz
miasto rok           
Łódź   2023       120
       2024       145
Kraków 2023       200
       2024       230
MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           names=['miasto', 'rok'])


In [24]:
# I z powrotem - reset_index:
df2 = df_hier.reset_index()
print(df2)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


### Tabele przestawne (ang. _Pivot Table_)

In [27]:
df2_my_pivot = df.set_index(["miasto", "rok"])["sprzedaz"].unstack()
print(df2_my_pivot)

rok     2023  2024
miasto            
Kraków   200   230
Łódź     120   145


In [28]:
df_pivot = pd.pivot_table(df,
                         index="miasto",
                         columns="rok",
                         values="sprzedaz")
print(df_pivot)

rok      2023   2024
miasto              
Kraków  200.0  230.0
Łódź    120.0  145.0


In [29]:
df_kwartaly = pd.DataFrame({
    "miasto":   ["Łódź"]*4 + ["Kraków"]*4,
    "rok":      [2023]*4 + [2023]*4,
    "kwartal":  ["Q1","Q2","Q3","Q4"] * 2,
    "sprzedaz": [30, 25, 35, 30, 50, 45, 55, 50]
})

df_kwartaly_sum = pd.pivot_table(df_kwartaly,
                                index="miasto",
                                columns="rok",
                                values="sprzedaz",
                                aggfunc="sum")
print(df_kwartaly_sum)

rok     2023
miasto      
Kraków   200
Łódź     120


### Agregacje po poziomie

In [30]:
sprzedaz.index.names = ["miasto", "rok"]
sprzedaz.groupby(level="miasto").sum()

,0
miasto,
Kraków,430
Łódź,265


In [31]:
sprzedaz.groupby(level="rok").mean()

,0
rok,
2023,160.0
2024,187.5


### Agregacja w DataFrame z MultiIndex

In [32]:
frame = pd.DataFrame({
    "sprzedaz": [120, 145, 200, 230],
    "koszty":   [80,  90,  130, 140]
}, index=[["Łódź","Łódź","Kraków","Kraków"],
          [2023,  2024,  2023,    2024]])
frame.index.names = ["miasto", "rok"]
frame.groupby(level="miasto").sum()

,sprzedaz,koszty
miasto,,
Kraków,430,270
Łódź,265,170


In [33]:
frame.groupby(level="miasto").agg(["sum", "mean"])

sprzedaz        koszty       
            sum   mean    sum   mean
miasto                              
Kraków      430  215.0    270  135.0
Łódź        265  132.5    170   85.0

In [34]:
frame["sprzedaz"].groupby(level="miasto").agg(
    total="sum", srednia="mean", rozstep=lambda x: x.max()-x.min()
    )

,total,srednia,rozstep
miasto,,,
Kraków,430,215.0,30
Łódź,265,132.5,25


# *Zadania*

Sieć "ModaŁódź" ma sklepy w trzech miastach. Każdy sklep sprzedaje trzy kategorie produktów. Dane obejmują 4 kwartały 2023 i 2024 roku.

## **Zadanie 1 — Budowanie MultiIndex**

a) Utwórz ramkę `df_hi` z hierarchicznym indeksem (`miasto`, `kategoria`, `rok`, `kwartal`). Posortuj indeks.

b) Wyświetl liczbę poziomów indeksu i ich nazwy.

c) Ile unikalnych kombinacji indeksu istnieje? Użyj `.index` do odpowiedzi.

In [52]:
df = pd.read_csv("/content/sklepy_moda.csv")
df_hi = df.set_index(["miasto", "kategoria", "rok", "kwartal"])
df_hi.sort_index(inplace=True)
print(f'liczba poziomow indeksu - {df_hi.index.nlevels}, nazwy {df_hi.index.names}')
print(f'unikalnych kombinacji {len(df_hi.index)}')

liczba poziomow indeksu - 4, nazwy ['miasto', 'kategoria', 'rok', 'kwartal']
unikalnych kombinacji 72


## **Zadanie 2 — Selekcje na MultiIndex**

a) Wybierz wszystkie dane dla Łodzi.
    
b) Wybierz dane dla Łodzi, kategorii Damska.
    
c) Wybierz dane dla wszystkich miast, ale tylko rok 2024. (Wskazówka: użyj metodę `xs()`)

d) Wybierz Q4 z obu lat, ale tylko dla Krakowa i kategorii Męska.

In [53]:
df_hi.loc["Łódź"]

Unnamed: 0  przychod_tys  koszt_tys  sztuki
kategoria rok  kwartal                                             
Damska    2023 Q1                0          50.5       35.2     615
               Q2                1          59.3       34.5     539
               Q3                2          61.9       34.3     915
               Q4                3          81.8       58.6     775
          2024 Q1                4          49.5       30.2     577
               Q2                5          62.5       39.8     627
               Q3                6          53.3       32.4     563
               Q4                7          85.8       55.0    1157
Dziecięca 2023 Q1               16          20.6       14.7     216
               Q2               17          31.4       19.0     370
               Q3               18          33.1       18.7     493
               Q4               19          43.6       30.7     409
          2024 Q1               20          35.4       24.9     301
               Q2               21          42.5       26.4     374
               Q3               22          35.7       22.0     301
               Q4               23          54.0       33.1     554
Męska     2023 Q1                8          40.3       26.9     335
               Q2                9          42.9       28.8     394
               Q3               10          42.0       28.8     465
               Q4               11          63.5       36.5     728
          2024 Q1               12          47.3       29.0     550
               Q2               13          47.9       31.6     445
               Q3               14          52.3       29.7     490
               Q4               15          71.1       39.7     730

In [55]:
df_hi.loc[("Łódź", "Damska")]

Unnamed: 0  przychod_tys  koszt_tys  sztuki
rok  kwartal                                             
2023 Q1                0          50.5       35.2     615
     Q2                1          59.3       34.5     539
     Q3                2          61.9       34.3     915
     Q4                3          81.8       58.6     775
2024 Q1                4          49.5       30.2     577
     Q2                5          62.5       39.8     627
     Q3                6          53.3       32.4     563
     Q4                7          85.8       55.0    1157

In [57]:
df_hi.xs(2024, level='rok')

Unnamed: 0  przychod_tys  koszt_tys  sztuki
miasto  kategoria kwartal                                             
Kraków  Damska    Q1               28          73.0       44.7     843
                  Q2               29          89.2       65.3     869
                  Q3               30          89.0       53.0     759
                  Q4               31         115.3       70.1    1052
        Dziecięca Q1               44          48.1       28.7     629
                  Q2               45          56.7       35.4     704
                  Q3               46          50.6       28.7     700
                  Q4               47          79.7       48.9     741
        Męska     Q1               36          60.5       38.3     578
                  Q2               37          74.4       42.7     771
                  Q3               38          67.2       43.9     868
                  Q4               39         100.3       62.5    1484
Wrocław Damska    Q1               52          58.3       42.8     824
                  Q2               53          76.1       45.8     960
                  Q3               54          71.1       46.6     689
                  Q4               55         107.6       61.2    1536
        Dziecięca Q1               68          36.4       24.2     416
                  Q2               69          52.6       31.0     686
                  Q3               70          35.5       26.2     521
                  Q4               71          61.4       45.0     650
        Męska     Q1               60          53.3       29.4     486
                  Q2               61          61.2       40.4     786
                  Q3               62          52.4       36.3     506
                  Q4               63          87.5       53.8    1157
Łódź    Damska    Q1                4          49.5       30.2     577
                  Q2                5          62.5       39.8     627
                  Q3                6          53.3       32.4     563
                  Q4                7          85.8       55.0    1157
        Dziecięca Q1               20          35.4       24.9     301
                  Q2               21          42.5       26.4     374
                  Q3               22          35.7       22.0     301
                  Q4               23          54.0       33.1     554
        Męska     Q1               12          47.3       29.0     550
                  Q2               13          47.9       31.6     445
                  Q3               14          52.3       29.7     490
                  Q4               15          71.1       39.7     730

In [59]:
df_hi.loc[('Kraków', 'Męska', slice(None), 'Q4')]

,Unnamed: 0,przychod_tys,koszt_tys,sztuki
rok,,,,
2023,35,89.4,63.8,1253
2024,39,100.3,62.5,1484


## **Zadanie 3 — pivot_table: roczne podsumowanie**

a) Utwórz tabelę przestawną `tab_miasta`, która pokaże **sumę przychodów** w wierszach per miasto, w kolumnach per rok.

b) Dodaj do tabeli kolumnę `zmiana_proc` — procentową zmianę przychodu między 2023 a 2024. Które miasto rosło najszybciej?

c) Utwórz drugą tabelę przestawną, w której wiersze to (miasto, kategoria), kolumny to rok, wartości to *średni przychód kwartalny*. Która kombinacja (miasto, kategoria) ma najwyższy średni przychód w 2024?

In [60]:
tab_miasta = pd.pivot_table(df_hi, index="miasto", columns="rok", values="przychod_tys", aggfunc="sum")
print(tab_miasta)

rok       2023   2024
miasto               
Kraków   824.3  904.0
Wrocław  697.7  753.4
Łódź     570.9  637.3


In [61]:
tab_miasta['zmiana_proc'] = ((tab_miasta[2024] - tab_miasta[2023]) / tab_miasta[2023]) * 100
print(tab_miasta)

rok       2023   2024  zmiana_proc
miasto                            
Kraków   824.3  904.0     9.668810
Wrocław  697.7  753.4     7.983374
Łódź     570.9  637.3    11.630758


In [68]:
tab_miasto_kategoria = pd.pivot_table(df_hi, index=["miasto", "kategoria"], columns="rok", values="przychod_tys", aggfunc="mean")
print(tab_miasto_kategoria)

rok                  2023    2024
miasto  kategoria                
Kraków  Damska     89.100  91.625
        Dziecięca  46.700  58.775
        Męska      70.275  75.600
Wrocław Damska     70.775  78.275
        Dziecięca  41.600  46.475
        Męska      62.050  63.600
Łódź    Damska     63.375  62.775
        Dziecięca  32.175  41.900
        Męska      47.175  54.650
---_-----najwyzszy sredni przychod rok
2023    89.100
2024    91.625
Name: (Kraków, Damska), dtype: float64


## **Zadanie 4 — Agregacja po poziomach**
a) Na `df_hi` oblicz sumę przychodów per miasto (agreguj po poziomie `"miasto"`).

b) Oblicz **średni przychód kwartalny per kategoria** (agreguj po poziomie `"kategoria"`).

c) Użyj `.agg(["sum", "mean", "max"])` na kolumnie `przychod_tys` pogrupowanej po `(miasto, rok)`. Który wiersz ma najwyższą wartość `max`?

In [69]:
suma_miasto = df_hi.groupby(level="miasto")["przychod_tys"].sum()
srednia_kategoria = df_hi.groupby(level="kategoria")["przychod_tys"].mean()
agregacje_miasto_rok = df_hi.groupby(level=["miasto", "rok"])["przychod_tys"].agg(["sum", "mean", "max"])

wiersz_max = agregacje_miasto_rok["max"].idxmax()
print(wiersz_max)

('Kraków', np.int64(2023))


## **Zadanie 5 — Marża i ranking**
a) W `df_hi` dodaj kolumnę `marza_tys = przychod_tys − koszt_tys`.

b) Utwórz tabelę przestawną: *wiersze = miasto*, *kolumny = kategoria*, *wartości = suma marży*. Które miasto jest najbardziej zyskowne? Która kategoria generuje najwyższą marżę?

In [70]:
df_hi["marza_tys"] = df_hi["przychod_tys"] - df_hi["koszt_tys"]
tab_marza = df_hi.pivot_table(index="miasto", columns="kategoria", values="marza_tys", aggfunc="sum")

miasto_najbardziej_zyskowne = tab_marza.sum(axis=1).idxmax()
kategoria_najwyzsza_marza = tab_marza.sum(axis=0).idxmax()

print(miasto_najbardziej_zyskowne)
print(kategoria_najwyzsza_marza)

Kraków
Damska
